# Modulo 1 · Lezione 8 — Gestione dei file

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | File di testo | Leggere e scrivere file `.txt` con `open()` |
| 2 | Modalità di apertura | `r`, `w`, `a`, `x` e le loro varianti binarie |
| 3 | `with` | Il gestore di contesto che chiude il file automaticamente |
| 4 | CSV | Leggere e scrivere dati tabulari con il modulo `csv` |
| 5 | JSON | Serializzare e deserializzare dati strutturati con `json` |
| 6 | Percorsi e `pathlib` | Lavorare con i percorsi in modo robusto e multipiattaforma |
| 7 | Pratica | Analizzatore CSV |

> 💡 **Perché è importante?** Quasi tutta l'IA e l'analisi dati inizia leggendo un file. I dataset sono CSV o JSON. I modelli si salvano su disco. I log sono file di testo. Saper leggere e scrivere file è un'abilità fondamentale.

---
## Preparazione — Creiamo i file di esempio

Esegui questa cella per prima per generare i file che useremo durante la lezione.

In [ ]:
import os

# Directory di lavoro per questa lezione
os.makedirs("file_lezione8", exist_ok=True)

# File di testo di esempio
with open("file_lezione8/poesia.txt", "w", encoding="utf-8") as f:
    f.write("""Viandante, sono le tue orme
il cammino e nulla più;
viandante, non c'è cammino,
il cammino si fa camminando.
Camminando si fa il cammino
e voltando lo sguardo indietro
si vede il sentiero che mai più
si tornerà a calpestare.
Viandante, non c'è cammino,
solo scie nel mare.
""")

# CSV di esempio: vendite
with open("file_lezione8/vendite.csv", "w", encoding="utf-8") as f:
    f.write("""data,prodotto,categoria,quantita,prezzo_unitario,venditore
2024-01-03,Laptop,Elettronica,2,899.99,Anna
2024-01-05,Mouse,Elettronica,10,25.50,Luigi
2024-01-08,Sedia,Mobili,3,199.00,Anna
2024-01-12,Monitor,Elettronica,1,349.00,Maria
2024-01-15,Tastiera,Elettronica,5,79.99,Luigi
2024-01-18,Tavolo,Mobili,2,299.00,Maria
2024-01-20,Laptop,Elettronica,1,899.99,Anna
2024-01-22,Cuffie,Elettronica,8,59.99,Luigi
2024-01-25,Scaffale,Mobili,4,149.00,Anna
2024-02-01,Monitor,Elettronica,2,349.00,Maria
2024-02-05,Mouse,Elettronica,15,25.50,Luigi
2024-02-10,Sedia,Mobili,6,199.00,Anna
2024-02-14,Laptop,Elettronica,3,899.99,Maria
2024-02-18,Tastiera,Elettronica,7,79.99,Luigi
2024-02-22,Tavolo,Mobili,1,299.00,Anna
""")

# JSON di esempio: configurazione
import json
config = {
    "app": "AnalizzatoreCSV",
    "version": "1.0",
    "configurazione": {
        "separatore": ",",
        "encoding": "utf-8",
        "decimali": 2
    },
    "colonne_numeriche": ["quantita", "prezzo_unitario"],
    "attivo": True
}
with open("file_lezione8/config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=4, ensure_ascii=False)

print("✅ File creati in file_lezione8/")
print(os.listdir("file_lezione8"))

---
## 1. File di testo — `open()`

La funzione `open()` apre un file e restituisce un **oggetto file** con cui possiamo leggere o scrivere.

```python
file = open("percorso/al/file.txt", modo, encoding="utf-8")
# ... operazioni ...
file.close()   # bisogna SEMPRE chiudere il file
```

### Modalità di apertura

| Modo | Nome | Cosa fa | Se il file non esiste |
|------|--------|----------|-------------------------|
| `"r"` | Read | Sola lettura | Errore (`FileNotFoundError`) |
| `"w"` | Write | Scrittura — **cancella** il contenuto esistente | Lo crea |
| `"a"` | Append | Aggiunge in fondo senza cancellare | Lo crea |
| `"x"` | Exclusive | Crea il file | Errore se esiste già |
| `"r+"` | Read+Write | Lettura e scrittura | Errore se non esiste |

> ⚠️ Aggiungi `"b"` per la modalità binaria: `"rb"`, `"wb"` — per immagini, PDF, ecc.

> 🌍 Usa sempre `encoding="utf-8"` per evitare problemi con caratteri speciali (accenti, lettere accentate…).

---
## 2. Il gestore di contesto `with`

Il modo corretto di aprire i file in Python è con `with`. Garantisce che il file venga chiuso automaticamente anche se si verifica un errore.

```python
# Senza with — pericoloso: se fallisce, il file resta aperto
file = open("dati.txt", "r")
contenuto = file.read()
file.close()   # potrebbe non arrivarci mai

# Con with — il file si chiude sempre all'uscita dal blocco
with open("dati.txt", "r", encoding="utf-8") as file:
    contenuto = file.read()
# qui il file è già chiuso
```

---
## 3. Lettura di file di testo

Ci sono tre metodi di lettura:

| Metodo | Restituisce | Quando usarlo |
|--------|----------|---------------|
| `.read()` | `str` con tutto il contenuto | File piccoli |
| `.readline()` | `str` con una sola riga | Riga per riga |
| `.readlines()` | `list` di stringhe (una per riga) | Quando serve la lista completa |
| Iterare il file | Una riga per iterazione | File grandi (non carica tutto in memoria) |

In [ ]:
# .read() — tutto il contenuto come una stringa
with open("file_lezione8/poesia.txt", "r", encoding="utf-8") as f:
    contenuto = f.read()

print(type(contenuto))
print(f"Caratteri: {len(contenuto)}")
print("---")
print(contenuto)

In [ ]:
# .readlines() — lista di righe (include il \n alla fine di ognuna)
with open("file_lezione8/poesia.txt", "r", encoding="utf-8") as f:
    righe = f.readlines()

print(f"Numero di righe: {len(righe)}")
print(f"Prima riga: {repr(righe[0])}")   # repr() mostra il \n

# Pulire il \n con strip()
righe_pulite = [l.strip() for l in righe]
print(f"Prima pulita: {repr(righe_pulite[0])}")

In [ ]:
# Iterare il file — più efficiente per file grandi
# (non carica tutto in memoria in una volta)
print("Righe che contengono 'cammino':")
with open("file_lezione8/poesia.txt", "r", encoding="utf-8") as f:
    for numero, riga in enumerate(f, start=1):
        if "cammino" in riga.lower():
            print(f"  Riga {numero}: {riga.strip()}")

---
## 4. Scrittura di file di testo

In [ ]:
# Modo "w" — crea o sovrascrive
with open("file_lezione8/note.txt", "w", encoding="utf-8") as f:
    f.write("Prima riga\n")
    f.write("Seconda riga\n")
    f.writelines(["Terza\n", "Quarta\n", "Quinta\n"])  # scrive una lista

# Verifica
with open("file_lezione8/note.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Modo "a" — aggiunge in fondo senza cancellare il contenuto
with open("file_lezione8/note.txt", "a", encoding="utf-8") as f:
    f.write("Sesta riga (aggiunta)\n")
    f.write("Settima riga (aggiunta)\n")

with open("file_lezione8/note.txt", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# Caso reale: scrivere un log di eventi
from datetime import datetime

def scrivi_log(percorso, livello, messaggio):
    """Aggiunge una voce al file di log."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    voce      = f"[{timestamp}] [{livello.upper()}] {messaggio}\n"
    with open(percorso, "a", encoding="utf-8") as f:
        f.write(voce)

log_path = "file_lezione8/app.log"
scrivi_log(log_path, "info",  "Applicazione avviata")
scrivi_log(log_path, "info",  "Caricamento dati...")
scrivi_log(log_path, "error", "File vendite_feb.csv non trovato")
scrivi_log(log_path, "info",  "Uso dei dati in cache")

with open(log_path, "r", encoding="utf-8") as f:
    print(f.read())

---
## 5. CSV — Comma-Separated Values

Un CSV è un file di testo dove ogni riga è una riga di dati e i valori sono separati da virgole (o un altro delimitatore). È il formato più usato per scambiare dati tabulari.

```
data,prodotto,quantita,prezzo
2024-01-03,Laptop,2,899.99
2024-01-05,Mouse,10,25.50
```

Python ha il modulo `csv` nella libreria standard — non serve installare nulla.

### Perché non semplicemente `.split(",")`?

Un CSV può contenere virgole dentro valori tra virgolette:
```
"Rossi, Anna",28,"Roma, Italia"
```
Il modulo `csv` gestisce questi casi correttamente; `.split(",")` no.

In [ ]:
import csv

# csv.reader — restituisce ogni riga come una lista di stringhe
print("=== csv.reader ===")
with open("file_lezione8/vendite.csv", "r", encoding="utf-8") as f:
    lettore = csv.reader(f)
    intestazione = next(lettore)   # prima riga = nomi delle colonne
    print(f"Intestazione: {intestazione}")
    print()
    for i, riga in enumerate(lettore):
        print(riga)
        if i == 2: break   # mostriamo solo le prime 3 righe

In [ ]:
# csv.DictReader — restituisce ogni riga come un dizionario
# La chiave è il nome della colonna — molto più comodo
print("=== csv.DictReader ===")
with open("file_lezione8/vendite.csv", "r", encoding="utf-8") as f:
    lettore = csv.DictReader(f)
    righe   = list(lettore)   # carichiamo tutte le righe

print(f"Righe totali: {len(righe)}")
print(f"Colonne:      {list(righe[0].keys())}")
print()

# Accesso per nome di colonna
for riga in righe[:3]:
    print(f"{riga['data']}  {riga['prodotto']:15} "
          f"x{riga['quantita']}  {float(riga['prezzo_unitario']):.2f}€")

In [ ]:
# csv.writer — scrivere un CSV
studenti = [
    {"nome": "Anna Rossi",   "voto": 8.5, "promosso": True},
    {"nome": "Luigi Bianchi","voto": 4.0, "promosso": False},
    {"nome": "Maria Verdi",  "voto": 7.0, "promosso": True},
]

# csv.writer
with open("file_lezione8/studenti.csv", "w", encoding="utf-8", newline="") as f:
    scrittore = csv.writer(f)
    scrittore.writerow(["nome", "voto", "promosso"])   # intestazione
    for a in studenti:
        scrittore.writerow([a["nome"], a["voto"], a["promosso"]])

# csv.DictWriter — più elegante quando hai dei dict
campi = ["nome", "voto", "promosso"]
with open("file_lezione8/studenti_dict.csv", "w",
          encoding="utf-8", newline="") as f:
    scrittore = csv.DictWriter(f, fieldnames=campi)
    scrittore.writeheader()    # scrive l'intestazione automaticamente
    scrittore.writerows(studenti)

# Verifica
with open("file_lezione8/studenti_dict.csv", "r", encoding="utf-8") as f:
    print(f.read())

> ⚠️ **`newline=""`** quando si scrivono CSV su Windows evita che vengano inserite righe vuote extra tra le righe. È una particolarità del modulo `csv` — mettilo sempre in scrittura.

---
## 6. JSON — JavaScript Object Notation

JSON è il formato standard per scambiare dati strutturati tra sistemi (API web, configurazioni, database NoSQL…). La sua sintassi è molto simile ai dizionari di Python.

```json
{
    "nome": "Anna",
    "eta": 28,
    "attivo": true,
    "etichette": ["python", "data"],
    "indirizzo": null
}
```

### Equivalenze Python ↔ JSON

| Python | JSON |
|--------|------|
| `dict` | object `{}` |
| `list`, `tuple` | array `[]` |
| `str` | string `""` |
| `int`, `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

### Le quattro funzioni del modulo `json`

| Funzione | Cosa fa | Direzione |
|---------|----------|----------|
| `json.dump(obj, f)` | Python → JSON su file | Python → Disco |
| `json.load(f)` | JSON su file → Python | Disco → Python |
| `json.dumps(obj)` | Python → stringa JSON | Python → Stringa |
| `json.loads(testo)` | Stringa JSON → Python | Stringa → Python |

In [ ]:
import json

# json.load — leggere un JSON da file
with open("file_lezione8/config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

print(type(config))           # dict
print(config["app"])          # AnalizzatoreCSV
print(config["version"])      # 1.0
print(config["configurazione"]["decimali"])   # 2
print(config["colonne_numeriche"])            # ['quantita', 'prezzo_unitario']
print()

# L'oggetto Python è già pronto per l'uso
for chiave, valore in config["configurazione"].items():
    print(f"  {chiave}: {valore}")

In [ ]:
# json.dump — scrivere un JSON su file
risultato_analisi = {
    "file": "vendite.csv",
    "totale_righe": 15,
    "totale_vendite": 12345.67,
    "prodotti": ["Laptop", "Mouse", "Sedia"],
    "completo": True,
    "errori": None
}

with open("file_lezione8/risultato.json", "w", encoding="utf-8") as f:
    json.dump(
        risultato_analisi,
        f,
        indent=4,            # indentazione leggibile
        ensure_ascii=False   # permette accenti e caratteri speciali
    )

# Verifica
with open("file_lezione8/risultato.json", "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
# json.dumps e json.loads — lavorare con stringhe (senza file)
# Molto utile quando ricevi dati da un'API o li invii

dati = {"nome": "Anna", "punteggio": 9.5, "promosso": True}

# Python → stringa JSON
testo_json = json.dumps(dati, ensure_ascii=False)
print(f"Stringa JSON: {testo_json}")
print(f"Tipo:         {type(testo_json)}")

# Stringa JSON → Python
recuperato = json.loads(testo_json)
print(f"\nRecuperato:   {recuperato}")
print(f"Tipo:         {type(recuperato)}")
print(f"Accesso:      {recuperato['nome']}")

---
## 7. `pathlib` — Percorsi in modo robusto

`pathlib` è il modo moderno di lavorare con i percorsi dei file in Python. Risolve il problema dei percorsi che funzionano su Mac/Linux (`/`) ma non su Windows (`\`).

```python
from pathlib import Path

percorso = Path("file_lezione8") / "vendite.csv"   # / costruisce i percorsi
```

In [ ]:
from pathlib import Path

# Costruire percorsi in modo multipiattaforma
cartella = Path("file_lezione8")
file = cartella / "vendite.csv"

print(f"Percorso:   {file}")
print(f"Nome:       {file.name}")
print(f"Estensione: {file.suffix}")
print(f"Senza est:  {file.stem}")
print(f"Cartella:   {file.parent}")
print(f"Esiste?     {file.exists()}")
print(f"È file?     {file.is_file()}")
print(f"Dimensione: {file.stat().st_size} bytes")

# Elencare i file di una directory
print("\nFile in file_lezione8/:")
for f in sorted(cartella.iterdir()):
    print(f"  {f.name:30} {f.stat().st_size:6} bytes")

In [ ]:
# pathlib integra lettura e scrittura direttamente
percorso = Path("file_lezione8") / "nota_veloce.txt"

# Scrivere direttamente con Path
percorso.write_text("Questa è una nota veloce.\n", encoding="utf-8")

# Leggere direttamente con Path
contenuto = percorso.read_text(encoding="utf-8")
print(contenuto)

# Cercare file per pattern
print("File CSV:")
for csv_file in Path("file_lezione8").glob("*.csv"):
    print(f"  {csv_file.name}")

---
## 8. Gestione degli errori con i file

I file sono una fonte comune di errori: il file non esiste, non abbiamo i permessi, il disco è pieno… Dobbiamo sempre anticiparli.

In [ ]:
# Gli errori più comuni con i file
from pathlib import Path

def leggi_file_sicuro(percorso):
    """Legge un file e gestisce gli errori più comuni."""
    try:
        with open(percorso, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        print(f"Errore: il file '{percorso}' non esiste.")
        return None
    except PermissionError:
        print(f"Errore: non hai il permesso di leggere '{percorso}'.")
        return None
    except UnicodeDecodeError:
        print(f"Errore: il file '{percorso}' non è in UTF-8. Prova con encoding='latin-1'.")
        return None

# Prova con file esistente
contenuto = leggi_file_sicuro("file_lezione8/note.txt")
if contenuto:
    print(f"Letto: {len(contenuto)} caratteri")

# Prova con file inesistente
leggi_file_sicuro("non_esiste.txt")

---
## 9. Pratica — Analizzatore CSV

Costruiamo un analizzatore del file `vendite.csv` che abbiamo generato all'inizio. Applichiamo tutto ciò che abbiamo imparato: leggere CSV, elaborare dati, salvare risultati in JSON e generare un report in testo.

---

### Esercizio 1 — Caricare e pulire il CSV

**Consegna:** Scrivi una funzione `carica_vendite(percorso)` che:
1. Legga il CSV con `DictReader`.
2. Converta `quantita` in `int` e `prezzo_unitario` in `float`.
3. Aggiunga una colonna calcolata `totale` = `quantita × prezzo_unitario`.
4. Restituisca la lista di dizionari pulita.

**Output atteso (prime 3 righe):**
```
{'data': '2024-01-03', 'prodotto': 'Laptop', 'categoria': 'Elettronica',
 'quantita': 2, 'prezzo_unitario': 899.99, 'venditore': 'Anna', 'totale': 1799.98}
```

In [ ]:
# ✏️ IL TUO CODICE QUI
import csv
from pathlib import Path

PERCORSO_CSV = Path("file_lezione8") / "vendite.csv"


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
import csv
from pathlib import Path

PERCORSO_CSV = Path("file_lezione8") / "vendite.csv"

def carica_vendite(percorso):
    """Carica il CSV delle vendite, converte i tipi e aggiunge la colonna 'totale'."""
    vendite = []
    with open(percorso, "r", encoding="utf-8") as f:
        for riga in csv.DictReader(f):
            riga["quantita"]        = int(riga["quantita"])
            riga["prezzo_unitario"] = float(riga["prezzo_unitario"])
            riga["totale"]          = round(riga["quantita"] * riga["prezzo_unitario"], 2)
            vendite.append(riga)
    return vendite

vendite = carica_vendite(PERCORSO_CSV)
print(f"Righe caricate: {len(vendite)}")
for v in vendite[:3]:
    print(v)
```

**Punti chiave:**
- `DictReader` restituisce tutti i valori come stringhe — la conversione dei tipi è il primo passo obbligatorio.
- Modifichiamo il dict direttamente nel ciclo: `riga["quantita"] = int(...)` — non serve creare un nuovo dict.
- `round(..., 2)` evita errori di virgola mobile nella colonna `totale`.
</details>

---
### Esercizio 2 — Statistiche globali

**Consegna:** Scrivi una funzione `statistiche_globali(vendite)` che restituisca un dizionario con:
- `totale_righe`: numero di record
- `totale_fatturato`: somma di tutti i `totale`
- `scontrino_medio`: media del campo `totale`
- `vendita_massima`: la riga con il `totale` più alto (dict completo)
- `vendita_minima`: la riga con il `totale` più basso (dict completo)
- `prodotti_unici`: set di prodotti distinti
- `venditori`: set di venditori

**Output atteso:**
```
Righe totali:       15
Totale fatturato:   10.247,82 €
Scontrino medio:    683,19 €
Vendita massima:    Laptop × 3  →  2.699,97 € (Maria)
Prodotti unici:     6
```

In [ ]:
# ✏️ IL TUO CODICE QUI
# Usa la lista 'vendite' dell'esercizio precedente


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
def statistiche_globali(vendite):
    """Calcola statistiche generali del dataset di vendite."""
    totali = [v["totale"] for v in vendite]

    return {
        "totale_righe":      len(vendite),
        "totale_fatturato":  round(sum(totali), 2),
        "scontrino_medio":   round(sum(totali) / len(totali), 2),
        "vendita_massima":   max(vendite, key=lambda v: v["totale"]),
        "vendita_minima":    min(vendite, key=lambda v: v["totale"]),
        "prodotti_unici":    {v["prodotto"] for v in vendite},
        "venditori":         {v["venditore"] for v in vendite}
    }

stats = statistiche_globali(vendite)

max_v = stats["vendita_massima"]
min_v = stats["vendita_minima"]

print(f"Righe totali:       {stats['totale_righe']}")
print(f"Totale fatturato:   {stats['totale_fatturato']:,.2f} €")
print(f"Scontrino medio:    {stats['scontrino_medio']:,.2f} €")
print(f"Vendita massima:    {max_v['prodotto']} × {max_v['quantita']} "
      f" →  {max_v['totale']:,.2f} € ({max_v['venditore']})")
print(f"Vendita minima:     {min_v['prodotto']} × {min_v['quantita']} "
      f" →  {min_v['totale']:,.2f} € ({min_v['venditore']})")
print(f"Prodotti unici:     {len(stats['prodotti_unici'])} → {sorted(stats['prodotti_unici'])}")
print(f"Venditori:          {sorted(stats['venditori'])}")
```

**Punti chiave:**
- `{v["prodotto"] for v in vendite}` è una **set comprehension** — costruisce il set di prodotti unici in una riga.
- `max(vendite, key=lambda v: v["totale"])` restituisce il dict completo della vendita più alta — `key` indica il criterio di confronto.
- `{stats['totale_fatturato']:,.2f}` usa il formato `,` per il separatore delle migliaia e `.2f` per due decimali.
</details>

---
### Esercizio 3 — Analisi per raggruppamento

**Consegna:** Scrivi funzioni per raggruppare e riassumere le vendite:

1. `vendite_per_venditore(vendite)` → dict `{venditore: totale_fatturato}`
2. `vendite_per_categoria(vendite)` → dict `{categoria: totale_fatturato}`
3. `vendite_per_prodotto(vendite)` → dict `{prodotto: {"unita": int, "fatturato": float}}`

Mostra i risultati ordinati dal fatturato più alto al più basso.

**Output atteso:**
```
=== Per venditore ===
Maria   3.947,96 €
Anna    3.894,94 €
Luigi   2.404,92 €

=== Per categoria ===
Elettronica   8.533,86 €
Mobili        1.713,96 €
```

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
def vendite_per_venditore(vendite):
    risultato = {}
    for v in vendite:
        chiave = v["venditore"]
        risultato[chiave] = round(risultato.get(chiave, 0) + v["totale"], 2)
    return dict(sorted(risultato.items(), key=lambda x: x[1], reverse=True))

def vendite_per_categoria(vendite):
    risultato = {}
    for v in vendite:
        chiave = v["categoria"]
        risultato[chiave] = round(risultato.get(chiave, 0) + v["totale"], 2)
    return dict(sorted(risultato.items(), key=lambda x: x[1], reverse=True))

def vendite_per_prodotto(vendite):
    risultato = {}
    for v in vendite:
        chiave = v["prodotto"]
        if chiave not in risultato:
            risultato[chiave] = {"unita": 0, "fatturato": 0.0}
        risultato[chiave]["unita"]     += v["quantita"]
        risultato[chiave]["fatturato"] = round(
            risultato[chiave]["fatturato"] + v["totale"], 2)
    return dict(sorted(risultato.items(),
                       key=lambda x: x[1]["fatturato"], reverse=True))

# Mostrare i venditori
print("=== Per venditore ===")
for venditore, totale in vendite_per_venditore(vendite).items():
    print(f"  {venditore:<10} {totale:>10,.2f} €")

# Mostrare le categorie
print("\n=== Per categoria ===")
for cat, totale in vendite_per_categoria(vendite).items():
    print(f"  {cat:<15} {totale:>10,.2f} €")

# Mostrare i prodotti
print("\n=== Per prodotto ===")
for prod, dati in vendite_per_prodotto(vendite).items():
    print(f"  {prod:<15} {dati['unita']:>4} unità  {dati['fatturato']:>10,.2f} €")
```

**Punti chiave:**
- `risultato.get(chiave, 0)` restituisce 0 se la chiave non esiste ancora — evita `KeyError` ed elimina la necessità di inizializzare ogni voce.
- `sorted(risultato.items(), key=lambda x: x[1], reverse=True)` ordina il dict per valore dal più alto al più basso.
- `{venditore:<10}` allinea a sinistra su 10 caratteri; `{totale:>10,.2f}` allinea a destra con separatore delle migliaia.
</details>

---
### Esercizio 4 — Salvare i risultati in JSON e CSV

**Consegna:** Scrivi due funzioni:

1. `salva_riepilogo_json(stats, percorso)` — salva il dizionario di statistiche in un file JSON. Converte i `set` in liste ordinate per renderlo serializzabile.
2. `salva_riepilogo_csv(raggruppamento, percorso, colonne)` — salva uno qualsiasi dei dizionari di raggruppamento come CSV.

**File attesi:**
- `file_lezione8/riepilogo.json`
- `file_lezione8/vendite_per_venditore.csv`

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
import json
import csv
from pathlib import Path

def salva_riepilogo_json(stats, percorso):
    """Salva il dizionario di statistiche in JSON.
    Converte set e dict di vendite in tipi serializzabili.
    """
    def serializzabile(obj):
        """Converte tipi non serializzabili."""
        if isinstance(obj, set):
            return sorted(obj)
        if isinstance(obj, dict):
            return {k: serializzabile(v) for k, v in obj.items()}
        return obj

    riepilogo = {
        "totale_righe":      stats["totale_righe"],
        "totale_fatturato":  stats["totale_fatturato"],
        "scontrino_medio":   stats["scontrino_medio"],
        "prodotti_unici":    sorted(stats["prodotti_unici"]),
        "venditori":         sorted(stats["venditori"]),
        "vendita_massima":   dict(stats["vendita_massima"]),
        "vendita_minima":    dict(stats["vendita_minima"])
    }
    with open(percorso, "w", encoding="utf-8") as f:
        json.dump(riepilogo, f, indent=4, ensure_ascii=False)
    print(f"✅ Riepilogo salvato in {percorso}")

def salva_riepilogo_csv(raggruppamento, percorso, col_chiave, col_valore):
    """Salva un dict {chiave: valore} come CSV con due colonne."""
    with open(percorso, "w", encoding="utf-8", newline="") as f:
        scrittore = csv.writer(f)
        scrittore.writerow([col_chiave, col_valore])
        for chiave, valore in raggruppamento.items():
            scrittore.writerow([chiave, valore])
    print(f"✅ CSV salvato in {percorso}")

# Uso
stats = statistiche_globali(vendite)
salva_riepilogo_json(stats, Path("file_lezione8") / "riepilogo.json")

per_venditore = vendite_per_venditore(vendite)
salva_riepilogo_csv(
    per_venditore,
    Path("file_lezione8") / "vendite_per_venditore.csv",
    "venditore", "totale_fatturato"
)

# Verifica JSON
with open(Path("file_lezione8") / "riepilogo.json", "r", encoding="utf-8") as f:
    print(f.read())
```

**Punti chiave:**
- `json.dump` fallisce se ci sono tipi non serializzabili — i `set` di Python non esistono in JSON. Bisogna convertirli in liste con `sorted()`.
- La funzione `serializzabile()` annidata è un helper locale: serve solo a `salva_riepilogo_json`, quindi vive al suo interno.
- `newline=""` quando si apre il CSV per la scrittura è obbligatorio su Windows per evitare righe vuote.
</details>

---
### Esercizio 5 (sfida) — Report completo in testo

**Consegna:** Scrivi una funzione `genera_report(vendite, percorso_output)` che generi un report completo in testo semplice e lo salvi in `file_lezione8/report.txt`. Il report deve includere:

1. Intestazione con data e nome del file analizzato.
2. Statistiche globali.
3. Classifica dei venditori (dal fatturato più alto al più basso).
4. Dettaglio per prodotto (unità vendute e fatturato).
5. Piè di pagina con il totale delle sezioni generate.

**Anteprima del report:**
```
================================================
   REPORT VENDITE — 2024-03-01 10:30:15
================================================
File analizzato: vendite.csv
Record totali: 15

--- RIEPILOGO GLOBALE ---
Totale fatturato: 10.247,82 €
Scontrino medio:     683,19 €
...
```

In [ ]:
# ✏️ IL TUO CODICE QUI
from datetime import datetime


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
from datetime import datetime
from pathlib import Path

def genera_report(vendite, percorso_output):
    """Genera un report vendite completo in testo semplice."""
    stats          = statistiche_globali(vendite)
    per_venditore  = vendite_per_venditore(vendite)
    per_prodotto   = vendite_per_prodotto(vendite)
    per_categoria  = vendite_per_categoria(vendite)

    adesso = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    sep    = "=" * 50
    subsep = "-" * 50
    righe  = []

    # Intestazione
    righe += [
        sep,
        f"   REPORT VENDITE — {adesso}",
        sep,
        f"File analizzato: {PERCORSO_CSV.name}",
        f"Record totali:   {stats['totale_righe']}",
        f"Prodotti unici:  {len(stats['prodotti_unici'])}",
        f"Venditori:       {', '.join(sorted(stats['venditori']))}",
        ""
    ]

    # Riepilogo globale
    max_v = stats["vendita_massima"]
    min_v = stats["vendita_minima"]
    righe += [
        f"{'--- RIEPILOGO GLOBALE ':—<50}",
        f"Totale fatturato: {stats['totale_fatturato']:>10,.2f} €",
        f"Scontrino medio:  {stats['scontrino_medio']:>10,.2f} €",
        f"Vendita massima:  {max_v['prodotto']} ×{max_v['quantita']}"
        f"  =  {max_v['totale']:,.2f} € ({max_v['venditore']})",
        f"Vendita minima:   {min_v['prodotto']} ×{min_v['quantita']}"
        f"  =  {min_v['totale']:,.2f} € ({min_v['venditore']})",
        ""
    ]

    # Classifica venditori
    righe.append(f"{'--- CLASSIFICA VENDITORI ':—<50}")
    for rango, (venditore, totale) in enumerate(per_venditore.items(), start=1):
        righe.append(f"  {rango}. {venditore:<12} {totale:>10,.2f} €")
    righe.append("")

    # Dettaglio per categoria
    righe.append(f"{'--- PER CATEGORIA ':—<50}")
    for cat, totale in per_categoria.items():
        righe.append(f"  {cat:<20} {totale:>10,.2f} €")
    righe.append("")

    # Dettaglio per prodotto
    righe.append(f"{'--- DETTAGLIO PER PRODOTTO ':—<50}")
    righe.append(f"  {'Prodotto':<15} {'Unità':>9} {'Fatturato':>12}")
    righe.append(f"  {'-'*15} {'-'*9} {'-'*12}")
    for prod, dati in per_prodotto.items():
        righe.append(
            f"  {prod:<15} {dati['unita']:>9} {dati['fatturato']:>11,.2f} €"
        )
    righe.append("")

    # Piè di pagina
    righe += [
        sep,
        f"Report generato il {adesso}",
        f"Sezioni: Riepilogo globale | Venditori | Categorie | Prodotti",
        sep
    ]

    # Scrivere il file
    contenuto = "\n".join(righe)
    with open(percorso_output, "w", encoding="utf-8") as f:
        f.write(contenuto)

    print(f"✅ Report salvato in {percorso_output}")
    print(contenuto)

genera_report(vendite, Path("file_lezione8") / "report.txt")
```

**Punti chiave:**
- Accumulare le righe in una lista e unirle alla fine con `"\n".join(righe)` è più efficiente che concatenare stringhe in un ciclo.
- `f"{'--- RIEPILOGO GLOBALE ':—<50}"` usa il riempimento di formattazione: il testo si allinea a sinistra e il resto dei 50 caratteri viene riempito con `—`.
- Separare il calcolo (funzioni statistiche) dalla presentazione (report) segue il principio di responsabilità unica — se cambia il formato, non tocchiamo la logica.
- Il report si può leggere direttamente in un editor o inviare via email senza bisogno di software speciale.
</details>

---
## Riepilogo della lezione

### File di testo

```python
# Leggere
with open("percorso.txt", "r", encoding="utf-8") as f:
    tutto  = f.read()           # stringa completa
    righe  = f.readlines()      # lista di stringhe
    for riga in f: ...          # iterazione efficiente

# Scrivere
with open("percorso.txt", "w", encoding="utf-8") as f:   # sovrascrive
    f.write("testo\n")

with open("percorso.txt", "a", encoding="utf-8") as f:   # aggiunge
    f.write("altro testo\n")
```

### CSV

```python
import csv

# Leggere
with open("dati.csv", "r", encoding="utf-8") as f:
    for riga in csv.DictReader(f):     # riga è un dict
        print(riga["colonna"])

# Scrivere
with open("output.csv", "w", encoding="utf-8", newline="") as f:
    scrittore = csv.DictWriter(f, fieldnames=["a", "b"])
    scrittore.writeheader()
    scrittore.writerows(lista_di_dict)
```

### JSON

```python
import json

# Leggere da file
with open("dati.json", "r", encoding="utf-8") as f:
    dati = json.load(f)

# Scrivere su file
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(dati, f, indent=4, ensure_ascii=False)

# In memoria
testo = json.dumps(dati)     # Python → stringa
dati  = json.loads(testo)    # stringa → Python
```

### pathlib

```python
from pathlib import Path

percorso = Path("cartella") / "file.csv"   # costruire percorso
percorso.exists()                           # esiste?
percorso.stat().st_size                     # dimensione in byte
percorso.read_text(encoding="utf-8")        # leggere direttamente
percorso.write_text("testo", encoding="utf-8") # scrivere direttamente
Path("cartella").glob("*.csv")              # cercare per pattern
```

---

## Cosa vedremo nella prossima lezione?

Nella **Lezione 9** impareremo a gestire **errori ed eccezioni** in modo professionale:
- `try`, `except`, `else`, `finally`
- Tipi di eccezioni in Python
- Creare eccezioni personalizzate con `raise`

Con questo potremo far sì che l'analizzatore CSV gestisca file corrotti, colonne mancanti o dati malformati senza che il programma si rompa.